# Day 2 — Re-evaluate the distilled student (Kaggle T4)

Run AFTER `kaggle_distill.ipynb` finishes. Produces:

1. **Headline table for the DISTILLED student** at 5, 10, 20 DDIM steps with LPIPS.
2. **Method ablation row** — Day 1 teacher (5 steps) vs distilled student (5 steps).
3. **Step ablation for the student** — 5/10/20/50/100 steps.
4. **ARR grid** on the student.
5. **T4 efficiency** for the student (same params as teacher, so identical).
6. **Updated Figures 1 + 3.**

**Three datasets must be attached to this notebook:** LOL eval15, LOL-v2, AND `lumidiff-distill-ckpt` (the dataset you uploaded after `kaggle_distill.ipynb` finished). The Day 1 checkpoint dataset (e.g. `lumidiff-checkpoint`) should also be attached so we can run the teacher comparison.

Wall time: ~50–75 min.

In [ ]:
# === Cell 1: install ===
!pip install -q --upgrade pip
!pip install -q scikit-image lpips fvcore matplotlib

In [ ]:
# === Cell 2: clone repo ===
import os, sys, shutil, subprocess
BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

for f in ("kaggle_path_discovery.py", "make_teaser_figure.py", "make_step_ablation_figure.py"):
    if not os.path.exists(f):
        raise SystemExit(f"{f} missing in cloned repo. Push your latest commits.")
print("Repo ready.")

In [ ]:
# === Cell 3: PREFLIGHT — discover datasets ===
import kaggle_path_discovery as kpd
kpd.diagnose("/kaggle/input")
discoveries = kpd.discover_all("/kaggle/input")
kpd.print_picks(discoveries)

# === MANUAL OVERRIDE if needed ===
# discoveries['lolv2_real_test'] = {'low_dir': '...', 'high_dir': '...', 'n_images': 100, 'kind': 'test'}
# discoveries['eval15_test']     = {'low_dir': '...', 'high_dir': '...', 'n_images': 15,  'kind': 'test'}

if not discoveries.get('lolv2_real_test'):
    raise SystemExit("LOL-v2 Real Test split not found.")
if not discoveries.get('eval15_test'):
    print("NOTE: eval15 not found. eval15 rows will be skipped.")

EVAL15_LOW  = discoveries['eval15_test']['low_dir']  if discoveries.get('eval15_test') else None
EVAL15_HIGH = discoveries['eval15_test']['high_dir'] if discoveries.get('eval15_test') else None
LOLV2_LOW   = discoveries['lolv2_real_test']['low_dir']
LOLV2_HIGH  = discoveries['lolv2_real_test']['high_dir']

EVAL_RESULTS = "/kaggle/working/eval_results_distill"
os.makedirs(EVAL_RESULTS, exist_ok=True)

SPLITS = []
if EVAL15_LOW:
    SPLITS.append(f"eval15:{EVAL15_LOW}:{EVAL15_HIGH}")
SPLITS.append(f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}")
print("Splits:", SPLITS)

In [ ]:
# === Cell 4: locate the distilled student + the Day 1 teacher ===
import glob
all_ckpts = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
STUDENT_CKPT = next((c for c in all_ckpts if "distill" in c.lower() and "best" in c.lower()), None)
TEACHER_CKPT = next((c for c in all_ckpts if ("final" in c.lower() or "day1" in c.lower())
                                          and "distill" not in c.lower()), None)
if TEACHER_CKPT is None:
    # fallback: any .pth that's NOT a distilled or fine-tuned variant
    TEACHER_CKPT = next((c for c in all_ckpts if not any(
        k in c.lower() for k in ("distill", "_ft", "smoke")
    )), None)

# === MANUAL OVERRIDE ===
# STUDENT_CKPT = "/kaggle/input/lumidiff-distill-ckpt/best_distill_K5.pth"
# TEACHER_CKPT = "/kaggle/input/lumidiff-checkpoint/final.pth"

if STUDENT_CKPT is None:
    raise SystemExit(
        "best_distill_*.pth not found under /kaggle/input/. Upload it as a Kaggle dataset "
        "(e.g. 'lumidiff-distill-ckpt') and attach via right panel."
    )
if TEACHER_CKPT is None:
    raise SystemExit("Day 1 (teacher) checkpoint not found.")

print(f"STUDENT_CKPT = {STUDENT_CKPT}")
print(f"TEACHER_CKPT = {TEACHER_CKPT}")

In [ ]:
# === Cell 5: GPU sanity ===
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    raise SystemExit("Set Accelerator -> GPU T4 -> Save, then re-run.")

In [ ]:
# === Cell 6: HEADLINE — distilled student at 5, 10, 20 steps with LPIPS ===
import csv, sys, subprocess
for STEPS in (5, 10, 20):
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", str(STEPS),
        "--sampler", "ddim",
        "--checkpoint", STUDENT_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "headline_student"),
        "--tag", f"s{STEPS}",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# === Cell 7: BASELINE — Day 1 TEACHER at 5 steps (so we have the head-to-head) ===
cmd = [
    sys.executable, "evaluation.py",
    "--splits", *SPLITS,
    "--inference-steps", "5",
    "--sampler", "ddim",
    "--checkpoint", TEACHER_CKPT,
    "--results-root", os.path.join(EVAL_RESULTS, "baseline_teacher"),
    "--tag", "s5",
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 8: STEP ABLATION on the student ===
for N in (5, 10, 20, 50, 100):
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", str(N),
        "--sampler", "ddim",
        "--checkpoint", STUDENT_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "step_ablation_full"),
        "--tag", f"ddim_s{N}",
        "--no-lpips",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# === Cell 9: ARR grid on the student @ 5 steps ===
ARR_GRID = []
for ALPHA in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):
    tag = f"arr_a{int(ALPHA*100):03d}"
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}",
        "--inference-steps", "5",
        "--sampler", "ddim",
        "--checkpoint", STUDENT_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "arr_grid"),
        "--tag", tag,
        "--gate-alpha", str(ALPHA),
        "--gate-floor", "0.5",
        "--no-lpips",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)
    summary_path = os.path.join(EVAL_RESULTS, f"arr_grid_{tag}", "summary.csv")
    with open(summary_path) as f:
        for row in csv.DictReader(f):
            row["alpha"] = ALPHA
            ARR_GRID.append(row)

In [ ]:
# === Cell 10: T4 efficiency for the student ===
cmd = [
    sys.executable, "measure_efficiency.py",
    "--steps", "5", "10", "20",
    "--resolution", "400", "600",
    "--device", "cuda",
    "--checkpoint", STUDENT_CKPT,
    "--out-csv", os.path.join(EVAL_RESULTS, "efficiency_t4_student.csv"),
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 11: Render Figure 1 (teaser) and Figure 3 (step ablation) ===
subprocess.run([
    sys.executable, "make_teaser_figure.py",
    "--pred-s5",  os.path.join(EVAL_RESULTS, "headline_student_s5/lolv2_real"),
    "--pred-s20", os.path.join(EVAL_RESULTS, "headline_student_s20/lolv2_real"),
    "--low",      LOLV2_LOW,
    "--gt",       LOLV2_HIGH,
    "--per-image-csv",     os.path.join(EVAL_RESULTS, "headline_student_s5/per_image.csv"),
    "--per-image-csv-s20", os.path.join(EVAL_RESULTS, "headline_student_s20/per_image.csv"),
    "--split", "lolv2_real",
    "--out",     os.path.join(EVAL_RESULTS, "figure1_teaser_student.pdf"),
    "--out-png", os.path.join(EVAL_RESULTS, "figure1_teaser_student.png"),
], check=True)
subprocess.run([
    sys.executable, "make_step_ablation_figure.py",
    "--eval-root", EVAL_RESULTS,
    "--out",     os.path.join(EVAL_RESULTS, "figure3_step_ablation_student.pdf"),
    "--out-png", os.path.join(EVAL_RESULTS, "figure3_step_ablation_student.png"),
], check=True)

In [ ]:
# === Cell 12: print the four paper tables ===
def read_summary(path):
    if not os.path.exists(path): return []
    return list(csv.DictReader(open(path)))

def fmt(r, variant):
    lp = r.get('lpips_mean')
    lp_s = f"{float(lp):.4f}" if lp not in (None, '', 'None') else '-'
    return (f"| {variant} | {r['split']} | {r['n']} | {r['inference_steps']} | "
            f"{float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | {lp_s} | "
            f"{float(r['runtime_mean']):.3f} |")

print("\n### Table A — Headline (DISTILLED STUDENT)\n")
print("| Variant | Split | n | Steps | PSNR | SSIM | LPIPS | Latency/img (s) |")
print("|---|---|---|---|---|---|---|---|")
for tag in ("s5", "s10", "s20"):
    for r in read_summary(os.path.join(EVAL_RESULTS, f"headline_student_{tag}/summary.csv")):
        print(fmt(r, "distilled (ours)"))

print("\n### Table B — Method ablation (5 DDIM steps)\n")
print("| Variant | Split | n | Steps | PSNR | SSIM | LPIPS | Latency/img (s) |")
print("|---|---|---|---|---|---|---|---|")
for r in read_summary(os.path.join(EVAL_RESULTS, "baseline_teacher_s5/summary.csv")):
    print(fmt(r, "teacher (Day 1)"))
for r in read_summary(os.path.join(EVAL_RESULTS, "headline_student_s5/summary.csv")):
    print(fmt(r, "+ self-distillation (ours)"))

print("\n### Table C — Step ablation (distilled student)\n")
print("| Steps | Split | PSNR | SSIM | Latency/img (s) |")
print("|---|---|---|---|---|")
for N in (5, 10, 20, 50, 100):
    for r in read_summary(os.path.join(EVAL_RESULTS, f"step_ablation_full_ddim_s{N}/summary.csv")):
        print(f"| {N} | {r['split']} | {float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | {float(r['runtime_mean']):.3f} |")

print("\n### Table D — Adaptive Residual Rescaling (student, 5 DDIM steps, LOL-v2 Real)\n")
print("| alpha | PSNR | SSIM |")
print("|---|---|---|")
for r in ARR_GRID:
    print(f"| {float(r['alpha']):.2f} | {float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} |")

print("\n### Efficiency (T4, 400x600, student)\n")
with open(os.path.join(EVAL_RESULTS, "efficiency_t4_student.csv")) as f:
    print(f.read())

In [ ]:
# === Cell 13: bundle ===
import shutil
out_zip = "/kaggle/working/phase3_eval_distill_outputs.zip"
if os.path.exists(out_zip): os.remove(out_zip)
shutil.make_archive(out_zip.replace(".zip", ""), "zip", EVAL_RESULTS)
print("Wrote", out_zip)
print("Download via Kaggle right panel -> Output tab.")